In [ ]:
from sklearn.linear_model import LinearRegression,RANSACRegressor, HuberRegressor, TheilSenRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
#from xgboost import XGBRegressor
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from mlxtend.classifier import StackingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score


import numpy as np
import pandas as pd

class ML_Model:
    models = []
    def __init__(self, name, model, category, r2=0, mse=0, rmse=0, mae=0, ypred=[]):
        self.name = name
        self.model = model
        self.category = category
        self.rmse = rmse
        self.mae = mae
        self.r2 = r2
        self.mse = mse
        self.ypred = ypred
        ML_Model.models.append(self)

    def get_results(self):
        return {
            "R2 Score": self.r2,
            "MAE": self.mae,
            "MSE":self.mse,
            "RMSE": self.rmse,
            "Y-Pred": self.ypred
        }
    
    def get_error_metrics(models):
        error_metrics = {
            'Model': [],
            'R2 Score': [],
            'MAE': [],
            'MSE': [],
            'RMSE': []
        }
    
        for model in models:
            model_result = model.get_results()
            error_metrics['Model'].append(model.name)
            error_metrics['R2 Score'].append(model_result["R2 Score"])
            error_metrics['MAE'].append(model_result["MAE"])
            error_metrics['MSE'].append(model_result["MSE"])
            error_metrics['RMSE'].append(model_result["RMSE"])
        
        return error_metrics

    def get_model_names():
        return [model.name for model in ML_Model.models]


In [ ]:



lr = ML_Model("Linear Regression", LinearRegression(), "Linear")
ransac = ML_Model("RANSAC Regression", RANSACRegressor(), "Linear")
huber = ML_Model("Huber Regression", HuberRegressor(), "Linear")
tsr = ML_Model("Theil-Sen Regression", TheilSenRegressor(), "Linear")
dtr = ML_Model("Decision Tree", DecisionTreeRegressor(), "Tree")
rfr = ML_Model("Random Forest", RandomForestRegressor(), "Ensemble")
ada = ML_Model("AdaBoost", AdaBoostRegressor(), "Ensemble")
gdboost = ML_Model("Gradient Boost", GradientBoostingRegressor(), "Ensemble")
#xgboost = ML_Model("XGBoost", XGBRegressor(), "Ensemble")
knn = ML_Model("KNeighbors", KNeighborsRegressor(), "Neighbor")
svm = ML_Model("Support Vector Machine", SVR(), "SVM")


In [ ]:

# Load CSV into DataFrame and build feature/target arrays
data = pd.read_csv("mega_output.csv")
X = data.iloc[:, :-1].to_numpy()
Y = data.iloc[:, -1].to_numpy()
print(X)
print(Y)

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, shuffle=True)

base_learners = []

for ml_model in ML_Model.models:
    base_learners.append(ml_model.model)
    ml_model.model.fit(x_train, y_train)
    try:
        ml_model.ypred = ml_model.model.predict(x_test)
    except Exception as e:
        print("Error at model prediction: ", e)
    ml_model.r2 = r2_score(y_test, ml_model.ypred)
    ml_model.mse = mean_squared_error(y_test, ml_model.ypred)
    # Load CSV into DataFrame and build feature/target arrays
    data = pd.read_csv("mega_output.csv")
    X = data.iloc[:, :-1].to_numpy()
    Y = data.iloc[:, -1].to_numpy()
    ml_model.mae = mean_absolute_error(y_test, ml_model.ypred)
    ml_model.rmse = np.sqrt(ml_model.mse)
    print(f"""
        {ml_model.name}
        \n\tR2: {ml_model.r2}
        \n\tMAE: {ml_model.mae}
        \n\tMSE: {ml_model.mse}
        \n\tRMSE: {ml_model.rmse}
    """)


meta_model = LogisticRegression()
stacking_model = StackingClassifier(classifiers=base_learners, meta_classifier=meta_model, use_probas=True)

model_stack = stacking_model.fit(x_train, y_train)   
pred_stack = model_stack.predict(x_test)

stacked_r2 = r2_score(y_test, pred_stack)
stacked_mse = mean_squared_error(y_test, pred_stack)
stacked_mae = mean_absolute_error(y_test, pred_stack)
stacked_rmse = np.sqrt(stacked_mse)
print(f"""
    Stacking Model
    \n\tR2: {stacked_r2}
    \n\tMAE: {stacked_mae}
    \n\tMSE: {stacked_mse}
    \n\tRMSE: {stacked_rmse}
""")
